## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load Data

In [2]:
path = r'/content/drive/MyDrive/CVR College/Mini Project/notebooksV3/extracted_datasetV3.csv'
df = pd.read_csv(path)
df.head()

,Unnamed: 0,IDTRAIN_JALON,IDTRAIN,arrival_delay_time,teu_count,train_length [m],total_distance_trip [km],departure_delay_time [min],distance_between_stations [km],weight_per_length [t/m],weight_per_wagon [t/wagon]
0,0,3507,255,0,0.0,390.0,99,NaN,0.000000,906.666667,NaN
1,1,3508,255,60,0.0,390.0,99,60.0,84.338382,906.666667,NaN
2,2,3515,256,0,NaN,NaN,99,NaN,0.000000,NaN,NaN
3,3,3516,256,60,NaN,NaN,99,60.0,84.338382,NaN,NaN
4,4,3531,258,0,NaN,NaN,99,NaN,0.000000,NaN,NaN


In [4]:
df.shape

(61215, 11)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61215 entries, 0 to 61214
Data columns (total 11 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Unnamed: 0                      61215 non-null  int64  
 1   IDTRAIN_JALON                   61215 non-null  int64  
 2   IDTRAIN                         61215 non-null  int64  
 3   arrival_delay_time              61215 non-null  int64  
 4   teu_count                       61130 non-null  float64
 5   train_length [m]                61047 non-null  float64
 6   total_distance_trip [km]        61215 non-null  int64  
 7   departure_delay_time [min]      47000 non-null  float64
 8   distance_between_stations [km]  61215 non-null  float64
 9   weight_per_length [t/m]         60493 non-null  float64
 10  weight_per_wagon [t/wagon]      50532 non-null  float64
dtypes: float64(6), int64(5)
memory usage: 5.1 MB


In [3]:
# Missing values
df.isna().sum()

,0
Unnamed: 0,0
IDTRAIN_JALON,0
IDTRAIN,0
arrival_delay_time,0
teu_count,85
train_length [m],168
total_distance_trip [km],0
departure_delay_time [min],14215
distance_between_stations [km],0
weight_per_length [t/m],722


In [6]:
# Define numerical columns (excluding IDs)
num_cols = [
    'arrival_delay_time', 'teu_count', 'train_length [m]',
    'total_distance_trip [km]', 'departure_delay_time [min]',
    'distance_between_stations [km]', 'weight_per_length [t/m]',
    'weight_per_wagon [t/wagon]'
]

# ==========================================
# STEP 1: Median Imputation for Missing Values
# ==========================================
for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Filled missing values in {col} with median: {median_val}")

# ==========================================
# STEP 2: IQR Method for Outlier Removal
# ==========================================
print(f"\nOriginal shape: {df.shape}")

# Create a boolean mask to keep valid rows
valid_rows = pd.Series(True, index=df.index)

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Update the mask (keep rows within the bounds)
    col_mask = (df[col] >= lower_bound) & (df[col] <= upper_bound)
    valid_rows = valid_rows & col_mask

# Apply the mask to filter out the outliers
df_cleaned = df[valid_rows]

print(f"Shape after IQR outlier removal: {df_cleaned.shape}")

Filled missing values in teu_count with median: 0.0
Filled missing values in train_length [m] with median: 544.0
Filled missing values in departure_delay_time [min] with median: 60.0
Filled missing values in weight_per_length [t/m] with median: 1127.9411764705883
Filled missing values in weight_per_wagon [t/wagon] with median: 36523.80952380953

Original shape: (61215, 11)
Shape after IQR outlier removal: (38201, 11)


In [7]:
df_cleaned.describe()

,Unnamed: 0,IDTRAIN_JALON,IDTRAIN,arrival_delay_time,teu_count,train_length [m],total_distance_trip [km],departure_delay_time [min],distance_between_stations [km],weight_per_length [t/m],weight_per_wagon [t/wagon]
count,38201.000000,38201.000000,38201.000000,38201.000000,38201.000000,38201.000000,38201.000000,38201.000000,38201.000000,38201.000000,38201.000000
mean,32161.616921,72623.812387,13876.172770,26.927986,22.941252,581.449334,839.346064,44.401586,147.110223,1460.458951,40006.872105
std,17653.126973,30012.982703,5707.381518,28.463421,30.554268,84.425399,362.839363,24.125230,243.198839,611.815587,13968.913194
min,0.000000,3507.000000,255.000000,0.000000,0.000000,382.000000,0.000000,0.000000,0.000000,870.000000,16609.090909
25%,16545.000000,48031.000000,9128.000000,0.000000,0.000000,544.000000,457.000000,30.000000,0.000000,1029.411765,32941.176471
50%,33130.000000,74474.000000,14158.000000,15.000000,0.000000,544.000000,1045.000000,60.000000,40.031435,1127.941176,36523.809524
75%,47446.000000,97805.000000,19155.000000,60.000000,56.250000,680.000000,1137.000000,60.000000,151.301458,1911.158088,40345.290323
max,61214.000000,120917.000000,22417.000000,120.000000,98.360000,782.000000,1150.000000,90.000000,815.308053,3891.309524,76646.500000


In [8]:
# Save the cleaned dataset for the next steps
df_cleaned.to_parquet('rail_delay_cleaned.parquet')